# 03 MLP 分类

## 本节目标

- 用二维非线性数据理解 MLP 的作用。
- 使用 `nn.Sequential` 组织多层网络。
- 理解 logits、label 和 `CrossEntropyLoss` 的关系。
- 通过 loss 曲线和决策边界观察分类效果。

## 背景问题

本节关注的问题是：当两类数据不能用一条直线分开时，模型如何学习更灵活的分类边界？

MLP 通过多层线性变换和激活函数引入非线性表达能力。这个概念可以从代码角度理解为：线性层负责特征变换，ReLU 让模型不再只是一个线性函数。

## 核心概念

- 隐藏层：位于输入和输出之间的特征变换层。
- 激活函数：引入非线性，使模型可以拟合弯曲的边界。
- Logits：分类模型输出的原始分数，不需要提前 softmax。
- Accuracy：预测类别和真实标签一致的比例。

## 最小代码示例

下面先构造一个二维分类数据集。这个实验用于观察 MLP 是否能学习非线性决策区域。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

x, y = make_moons(n_samples=500, noise=0.18, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)

x_train = torch.tensor(x_train)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test, dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=18)
plt.title("Training data")
plt.grid(alpha=0.3)
plt.show()

## 完整实验

这里的关键不是网络层数越多越好，而是理解每一层在训练闭环中的位置。

### 训练与评估

训练时将 logits 直接传给 `CrossEntropyLoss`，标签使用类别索引。调试时可以优先检查 label dtype、logits shape，以及最后一层输出类别数是否和任务一致。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(120):
    model.train()
    logits = model(x_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()

print(f"final loss={losses[-1]:.4f}, test accuracy={test_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("MLP classification loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)
plt.show()

xx, yy = np.meshgrid(
    np.linspace(x_train[:, 0].min() - 0.5, x_train[:, 0].max() + 0.5, 150),
    np.linspace(x_train[:, 1].min() - 0.5, x_train[:, 1].max() + 0.5, 150),
)
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))
with torch.no_grad():
    zz = model(grid).argmax(dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
plt.scatter(x_test[:, 0], x_test[:, 1], c=y_test, cmap="coolwarm", s=18, edgecolor="k")
plt.title("Decision regions on test points")
plt.grid(alpha=0.3)
plt.show()

## 实验观察

从运行结果可以看到，loss 逐步下降，测试准确率通常较高。决策区域不再是一条简单直线，而是能贴合 moon 数据的弯曲结构，说明激活函数带来的非线性表达发挥了作用。

## 常见错误

- 把分类 label 设置成 float，导致 `CrossEntropyLoss` 报错。
- 在 logits 前手动 softmax，再传给 `CrossEntropyLoss`。
- 隐藏层没有激活函数，模型表达能力不足。
- 训练集和测试集使用了不一致的数据标准化方式。

初学者容易在这里混淆 logits 和概率。训练时关注 logits 即可，展示结果时再按需要转成概率。

In [ ]:
print("logits shape:", test_logits.shape)
print("label dtype:", y_train.dtype)
print("first logits:", test_logits[:3])

## 概念问答

**Q：为什么最后一层没有 ReLU？**  
A：分类 logits 可以是任意实数，后续由 `CrossEntropyLoss` 处理。

**Q：为什么不能先 softmax 再传给 `CrossEntropyLoss`？**  
A：`CrossEntropyLoss` 内部已经包含相关计算，提前 softmax 可能让训练变得不稳定或效果变差。

**Q：准确率高是否说明模型一定泛化好？**  
A：不一定。这里是小型实验，更可靠的判断需要更多数据、合理划分和重复实验。

## 小结

MLP 用激活函数把线性层组合成非线性模型。分类任务中最需要关注 logits、label dtype、loss 函数和输出类别数是否匹配。